In [1]:
import torch.nn as nn

In [5]:
import torch

In [2]:
type(nn)

module

In [ ]:
class googlenet(nn.Module):
    def __init__(self, in_features,num_classes_1,num_classes_2,num_classes_3,num_classes_4):
        super().__init__()
        # branch1 : k=1,s=1,p=0
        self.conv1 = nn.Conv2d(in_features,num_classes_1,kernel_size=1,stride=1,padding=0)
        # branch2 : k=1,s=1,p=0 -> k=3,s=1,p=1
        self.conv2_1 = nn.Conv2d(in_features,num_classes_2,kernel_size=1,stride=1,padding=0)
        self.conv2_2 = nn.Conv2d(num_classes_2,num_classes_2,kernel_size=3,stride=1,padding=1)
        # branch3 : k=1,s=1,p=0 -> k=5,s=1,p=2
        self.conv3_1 = nn.Conv2d(in_features,num_classes_3,kernel_size=1,stride=1,padding=0)
        self.conv3_2 = nn.Conv2d(num_classes_3,num_classes_3,kernel_size=5,stride=1,padding=2)

        # branch4 : pool(k=3,s=1,p=1) -> k=1,s=1,p=0
        self.maxpool = nn.MaxPool2d(kernel_size=3,stride=1,padding=1)
        self.conv4 = nn.Conv2d(in_features,num_classes_4,kernel_size=1,stride=1,padding=0)
        # batch normalization
        self.batchnorm2d1 = nn.BatchNorm2d(num_classes_1)
        self.bn2_1 = nn.BatchNorm2d(num_classes_2)
        self.bn2_2 = nn.BatchNorm2d(num_classes_2)
        self.bn3_1 = nn.BatchNorm2d(num_classes_3)
        self.bn3_2 = nn.BatchNorm2d(num_classes_3)
        self.batchnorm2d4 = nn.BatchNorm2d(num_classes_4)
        #relu
        self.relu = nn.ReLU()
    def forward(self,x):
        # branch1
        x1 = self.conv1(x)
        x1 = self.batchnorm2d1(x1)
        x1 = self.relu(x1)
        # branch2
        x2 = self.conv2_1(x)
        x2 = self.bn2_1(x2)
        x2 = self.relu(x2)
        x2 = self.conv2_2(x2)
        x2 = self.bn2_2(x2)
        x2 = self.relu(x2)
        # branch3
        x3 = self.conv3_1(x)
        x3 = self.bn3_1(x3)
        x3 = self.relu(x3)
        x3 = self.conv3_2(x3)
        x3 = self.bn3_2(x3)
        x3 = self.relu(x3)
        # branch4
        x4 = self.maxpool(x)
        x4 = self.conv4(x4)
        x4 = self.batchnorm2d4(x4)
        x4 = self.relu(x4)
        # concatenate
        out = torch.cat([x1,x2,x3,x4],dim=1)
        return out

In [6]:
from torchsummary import summary
model = googlenet(3, 192, 128, 96, 32)
summary(model,(3,224,224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1        [-1, 192, 224, 224]             768
       BatchNorm2d-2        [-1, 192, 224, 224]             384
              ReLU-3        [-1, 192, 224, 224]               0
            Conv2d-4        [-1, 128, 224, 224]             512
       BatchNorm2d-5        [-1, 128, 224, 224]             256
              ReLU-6        [-1, 128, 224, 224]               0
            Conv2d-7        [-1, 128, 224, 224]         147,584
       BatchNorm2d-8        [-1, 128, 224, 224]             256
              ReLU-9        [-1, 128, 224, 224]               0
           Conv2d-10         [-1, 96, 224, 224]             384
      BatchNorm2d-11         [-1, 96, 224, 224]             192
             ReLU-12         [-1, 96, 224, 224]               0
           Conv2d-13         [-1, 96, 224, 224]         230,496
      BatchNorm2d-14         [-1, 96, 2